## Gemini Statefull API

- Stateless: **generate_content()**  
- Pseudo-Statefull: **generate_content()+history**  
- Statefull: **chats()**  

pip install google-genai

In [1]:
import os, sys
from pprint import pprint
from importlib.metadata import version

from google import genai
from google.genai import types

print(f"{sys.version}")
print(f"google.genai={version('google-genai')}")

# ====== Gemini の設定 ======
# Gemini API Keyを取得
with open('GOOGLE_API_KEY.txt', 'r') as f:  # ファイルからAPI Keyを取得
    api_key = f.read().strip()
# Geminiクライアントの作成
GEMINI_CLIENT = genai.Client(api_key=api_key)

# Geminiモデルを指定
#GEMINI_LLM_MODEL = 'gemini-pro-latest'         # 賢い
#GEMINI_LLM_MODEL = 'gemini-flash-latest'       # まあ賢い
GEMINI_LLM_MODEL = 'gemini-flash-lite-latest'  # 安価

3.13.12 (tags/v3.13.12:1cbe481, Feb  3 2026, 18:22:25) [MSC v.1944 64 bit (AMD64)]
google.genai=1.68.0


### Stateless Query

**generate_content()**

単純な質問/回答。各質問間で情報は引き継がれない。

In [2]:
# LLMに質問への回答を生成させる。
def gen_answer(query):
    prompt = f'''
    質問に根拠付きで回答してください。
    情報が見つからない場合には「わかりません」と回答してください。
    
    質問: {query}
    '''
    try:
        response = GEMINI_CLIENT.models.generate_content(
            model=GEMINI_LLM_MODEL,
            contents=[prompt]
        )
        response = response.text
    except Exception as e:
        response = f"Error: {e}"

    return response

# 最初の質問
query_1st = 'いま渋谷にいるんだけど、近くの観光地を1つ教えて。'
response = gen_answer(query_1st)
print(f"\n## 質問: {query_1st}\n## 回答: \n{response}")

# 2番目の質問
query_2nd = 'そこへの行き方は？歩いて行ける？'
response = gen_answer(query_2nd)
print(f"\n## 質問: {query_2nd}\n## 回答: \n{response}")


## 質問: いま渋谷にいるんだけど、近くの観光地を1つ教えて。
## 回答: 
渋谷にいらっしゃるのですね。渋谷周辺には魅力的な観光地がたくさんあります。

**渋谷スクランブル交差点**はいかがでしょうか。

**根拠:**

1. **渋谷の象徴であること:** 渋谷スクランブル交差点は、世界的に有名な渋谷のランドマークであり、一度は訪れる価値のある観光スポットです。（出典：渋谷区観光協会など、一般的な観光情報）
2. **アクセス:** 渋谷駅のすぐ外にあり、現在いらっしゃる場所から最も近く、すぐにアクセスできます。
3. **体験:** 多数の人々が行き交う様子は圧巻で、写真撮影やカフェなどからその光景を眺めるだけでも観光体験として成立します。（出典：個人の観光体験談や観光ガイドブックなど）

もし、もう少し落ち着いた場所や文化的な場所をご希望の場合は、お知らせください。

## 質問: そこへの行き方は？歩いて行ける？
## 回答: 
ご質問ありがとうございます。

**「そこへの行き方は？歩いて行ける？」** というご質問に対して、**「そこ」が具体的にどこを指しているのか**が分かりかねます。

そのため、**現時点では具体的な行き方や徒歩でのアクセスが可能かどうかをお答えすることができません。**

**根拠:**

*   「そこ」という指示語が、文脈から特定の場所を指していることが確認できないため。

**もしよろしければ、「そこ」が指す場所（例：駅名、施設名、住所など）を教えていただけますでしょうか？** 場所が特定できれば、行き方や徒歩でのアクセス可能性についてお調べできます。


### Pseudo-Statefull Queries (Query Chain)

**generate_content()+history**  

過去の質問/回答(+推論過程)を次の質問に含めることにより、過去の推論状況を引き継ぐ。  
APIはステートレスであるが、疑似的にステートフルをな思考過程を実現する。  
累積された情報のAPIの使用量および通信コストが発生する。

In [3]:
# 1. 最初の回答を得る
query_1st = 'いま渋谷にいるんだけど、近くの観光地を1つ教えて。'
response = GEMINI_CLIENT.models.generate_content(
    model=GEMINI_LLM_MODEL,
    contents=query_1st
)
# モデルの回答（Contentオブジェクト）をそのまま保存しておく
# この中には「テキスト」だけでなく「思考シグネチャ」が含まれています
model_output_1 = response.candidates[0].content
print(f"\n## 質問: {query_1st}\n## 回答: \n{response.text}")

# 2. 履歴を含めて次のリクエストを送る
query_2nd = 'そこへの行き方は？歩いて行ける？'
history = [
    # --- 1回目のやり取り ---
    types.Content(role="user", parts=[types.Part(text=query_1st)]),
    model_output_1, # ← ここに thought_signature が入っていることが重要！

    # --- 2回目のやり取り ---
    types.Content(role="user", parts=[types.Part(text=query_2nd)])
]
response = GEMINI_CLIENT.models.generate_content(
    model=GEMINI_LLM_MODEL,
    contents=history
)
model_output_2 = response.candidates[0].content
print(f"\n## 質問: {query_2nd}\n## 回答: \n{response.text}")

# 3. 履歴を含めて次のリクエストを送る
query_3rd = '周辺でおすすめのランチは？'
history = [
    # --- 1回目のやり取り ---
    types.Content(role="user", parts=[types.Part(text=query_1st)]),
    model_output_1, # 1回目の回答（テキスト + 思考シグネチャ1）

    # --- 2回目のやり取り ---
    types.Content(role="user", parts=[types.Part(text=query_2nd)]),
    model_output_2, # 2回目の回答（テキスト + 思考シグネチャ2）

    # --- 3回目の新しい質問 ---
    types.Content(role="user", parts=[types.Part(text=query_3rd)])
]

final_response = GEMINI_CLIENT.models.generate_content(
    model=GEMINI_LLM_MODEL,
    contents=history
)
print(f"\n## 質問: {query_3rd}\n## 回答: \n{final_response.text}")


## 質問: いま渋谷にいるんだけど、近くの観光地を1つ教えて。
## 回答: 
渋谷にいらっしゃるのですね！渋谷周辺には魅力的な観光地がたくさんありますが、**「渋谷スクランブル交差点」**はいかがでしょうか？

世界的に有名な場所で、渋谷に来たならぜひ見ておきたいスポットです。

もし、もっと落ち着いた場所や、違うタイプの観光地をお探しでしたら、例えば以下のような選択肢もありますよ。

1. **渋谷スカイ (SHIBUYA SKY)**：渋谷上空から街を一望できる展望施設です。絶景を楽しめます。（スクランブル交差点からすぐ）
2. **明治神宮**：渋谷の喧騒から離れて、広大な森の中にある荘厳な神社です。散策に最適です。（原宿方面へ少し歩きます）
3. **渋谷PARCO（パルコ）**：最新のカルチャーやファッション、任天堂オフィシャルストアなどが入っているので、ショッピングやトレンドをチェックするのに良い場所です。

今、どちらの方面にいらっしゃるか、またはどんな気分か教えていただければ、よりぴったりの場所をご提案できます！

## 質問: そこへの行き方は？歩いて行ける？
## 回答: 
「渋谷スクランブル交差点」ですね！

**はい、歩いてすぐ行けます！**

もし今、あなたが**渋谷駅周辺**にいらっしゃるのであれば、スクランブル交差点は**渋谷駅の目の前**に広がっています。

### 行き方（渋谷駅周辺から）

1. **JRや私鉄の改札を出る**：
   * JR線、東急線、東京メトロ、京王線などの改札を出て「ハチ公口」方面を目指します。
2. **ハチ公口（または西口）を出る**：
   * 改札を出て広場に出ると、すぐ目の前に忠犬ハチ公の銅像があります。
3. **交差点へ**：
   * ハチ公像の向かい側（駅を出て正面）に、あの有名な巨大な交差点が広がっています。

**迷うことはまずないかと思います**が、もし現在地が駅の少し離れた場所でしたら、「渋谷スクランブル交差点の近くまで」と尋ねてみてください。

いってらっしゃいませ！たくさんの人で賑わっているのを楽しんできてくださいね。

## 質問: 周辺でおすすめのランチは？
## 回答: 
スクランブル交差点周辺は激戦区なので、美味しいランチの選択肢が無限にあります！

今のご気分に

### Statefull Queries

**chats()**

LLM側で、過去の質問/回答(+推論過程)キャッシュすることにより、過去の推論過程を引き継ぐ。  
ただし、LLM側で累積された情報のAPIの使用コストが発生する。

In [4]:
# chat_session を使用した実装
chat = GEMINI_CLIENT.chats.create(model=GEMINI_LLM_MODEL)

queries = [
    'いま渋谷にいるんだけど、近くの観光地を1つ教えて。',
    'そこへの行き方は？',
    '周辺でおすすめのランチは？'
]

for query in queries:
    try:
        # historyへの追加はSDKが内部で処理する
        response = chat.send_message(query)
        print(f"\n## 質問: {query}\n## 回答: \n{response.text}\n")
    except Exception as e:
        print(f"Error: {e}")
        break


## 質問: いま渋谷にいるんだけど、近くの観光地を1つ教えて。
## 回答: 
渋谷にいらっしゃるんですね！渋谷駅周辺には魅力的な観光スポットがたくさんありますが、すぐ近くで楽しめるおすすめを1つご紹介します。

**渋谷スカイ (SHIBUYA SKY)** はいかがでしょうか？

### 渋谷スカイ (SHIBUYA SKY) のおすすめポイント

*   **圧倒的な眺望:** 渋谷駅直結の渋谷スクランブルスクエアの屋上にある展望施設です。地上約230mからの東京の街並みを360度見渡せます。
*   **スクランブル交差点が見下ろせる:** 渋谷の代名詞であるスクランブル交差点を真上から見下ろすことができ、その迫力は圧巻です。
*   **夕方〜夜景が最高:** 特に夕暮れ時のマジックアワーから夜景にかけては、東京のきらめく景色が楽しめます。

もし天候が良く、少し高い場所から街全体を見渡したい気分でしたら、最高のスポットですよ！

---
**補足:**

*   現在、入場には**日時指定の事前予約**が推奨されています（当日券もありますが、混雑状況によります）。
*   もし天候が悪かったり、屋内の方がよろしければ、**渋谷スクランブル交差点**を少し離れた場所（スターバックスなど）から眺めるのも定番です。


## 質問: そこへの行き方は？
## 回答: 
「渋谷スカイ (SHIBUYA SKY)」は、渋谷のランドマークタワーである**渋谷スクランブルスクエア**の屋上にあるため、アクセスは非常に簡単です。

渋谷駅直結ですので、迷う心配はほとんどありません！

### 渋谷スカイへの行き方（渋谷駅直結）

1.  **渋谷スクランブルスクエアを目指す:**
    *   JR、東京メトロ、東急線などの**渋谷駅**から、**「東口」**方面へ向かってください。
    *   目指すべき建物は、渋谷駅の真上にある高層ビル **「渋谷スクランブルスクエア (SHIBUYA SCRAMBLE SQUARE)」** です。

2.  **エントランスへ:**
    *   ビルに入ったら、エスカレーターやエレベーターで**2階**へ上がってください。
    *   2階に入場チケットカウンターや、屋上へ向かうエレベーターの乗り場があります。
